# SECTION 1: Google Colab Setup & Drive Mount\nMounting Google Drive to ensure persistent storage of models and checkpoints.

In [ ]:
import os
import sys

# Google Drive Mount (Only executes if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_ROOT = '/content/drive/MyDrive/KokBisaResearch'
    os.makedirs(COLAB_ROOT, exist_ok=True)
    os.chdir(COLAB_ROOT)
    print(f"Mounted Google Drive. Working directory set to: {os.getcwd()}")
except ImportError:
    print("Not running in Google Colab. Using local environment.")
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    print(f"Working directory set to: {os.getcwd()}")


# SECTION 2: Environment Setup

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import math

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Available: True ({torch.cuda.get_device_name(0)})")
    device = 0
else:
    print(f"CUDA Available: False (CPU ONLY)")
    device = -1


# SECTION 3: Contract Validation & Paths

In [ ]:
CONTRACT_PATH = "config/pipeline_contract.json"

with open(CONTRACT_PATH, "r") as f:
    contract = json.load(f)

MODEL_DIR = contract["model_directory"]
LABEL_MAPPING_PATH = contract["label_mapping"]
CORPUS_PATH = contract["corpus_input"]
INFERENCE_OUTPUT = contract["inference_output"]
CHECKPOINT_DIR = contract["inference_checkpoint_dir"]
FIGURE_DIR = contract["figure_output_dir"]

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INFERENCE_OUTPUT), exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# Compatibility Check
assert os.path.exists(MODEL_DIR), f"Model directory {MODEL_DIR} not found!"
assert os.path.exists(LABEL_MAPPING_PATH), f"Label mapping {LABEL_MAPPING_PATH} not found!"
assert os.path.exists(CORPUS_PATH), f"Corpus {CORPUS_PATH} not found!"
print("Pipeline Contract Validated.")


# SECTION 4: Load Final Model & Tokenizer

In [ ]:
print("Loading Model and Tokenizer from:", MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

with open(LABEL_MAPPING_PATH, "r") as f:
    raw_mapping = json.load(f)
    
if "id2label" in raw_mapping:
    mapping_dict = raw_mapping["id2label"]
    id2label = {int(k): v for k, v in mapping_dict.items()}
elif "label2id" in raw_mapping:
    mapping_dict = raw_mapping["label2id"]
    id2label = {int(v): k for k, v in mapping_dict.items()}
else:
    id2label = {int(v): k for k, v in raw_mapping.items()}

# Set up pipeline
classifier = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer, 
    device=device,
    top_k=None # Ensures it returns a list of all scores per sequence
)
print("Pipeline ready.")


# SECTION 5: Load Full Corpus

In [ ]:
df_corpus = pd.read_parquet(CORPUS_PATH)
print(f"Loaded corpus with {len(df_corpus):,} records.")

text_col = "cleaned_text" if "cleaned_text" in df_corpus.columns else "text"
df_corpus[text_col] = df_corpus[text_col].fillna("").astype(str)

# Ensure ID col exists
if "comment_id" not in df_corpus.columns:
    df_corpus["comment_id"] = [f"C_{i}" for i in range(len(df_corpus))]


# SECTION 6: Robust Inference Checkpointing\nUsing chunked batching to prevent RAM exhaustion and allow resuming on disconnect.

In [ ]:
import glob

CHUNK_SIZE = 10000
BATCH_SIZE = 128  # HF Pipeline batch size

total_chunks = math.ceil(len(df_corpus) / CHUNK_SIZE)
print(f"Total Chunks to process: {total_chunks} (Chunk size: {CHUNK_SIZE})")

processed_ids = set()
checkpoint_files = glob.glob(os.path.join(CHECKPOINT_DIR, "batch_*.parquet"))

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} existing checkpoints. Loading to resume...")
    for cp in checkpoint_files:
        df_cp = pd.read_parquet(cp)
        processed_ids.update(df_cp["comment_id"].tolist())
    print(f"Resuming. {len(processed_ids):,} comments already processed.")
else:
    print("No checkpoints found. Starting from scratch.")

df_to_process = df_corpus[~df_corpus["comment_id"].isin(processed_ids)].copy()
print(f"Remaining comments to process: {len(df_to_process):,}")


# SECTION 7: Execution Loop

In [ ]:
import time

# We will process df_to_process in chunks of CHUNK_SIZE
chunks = [df_to_process[i:i + CHUNK_SIZE] for i in range(0, len(df_to_process), CHUNK_SIZE)]

for chunk_idx, chunk in enumerate(chunks):
    start_time = time.time()
    texts = chunk[text_col].tolist()
    ids = chunk["comment_id"].tolist()
    
    print(f"\nProcessing Chunk {chunk_idx + 1}/{len(chunks)} ({len(texts)} comments)...")
    
    # Run pipeline with batching
    # Pipeline returns a list of lists of dicts if return_all_scores=True
    predictions = classifier(texts, batch_size=BATCH_SIZE, truncation=True, max_length=128)
    
    predicted_labels = []
    confidences = []
    margins = []
    
    for pred in predictions:
        # Compatibility handling: newer transformers return a single dict if top_k is not respected
        if isinstance(pred, dict):
            pred = [pred]
            
        # Sort by score desc
        pred.sort(key=lambda x: x["score"], reverse=True)
        top1 = pred[0]
        top2_score = pred[1]["score"] if len(pred) > 1 else 0.0
        
        # Label string comes as LABEL_0 etc. We need to map it back using our mapping or id2label
        # If the pipeline uses config.id2label, it might already map it if configured properly during save.
        # Let's extract the integer ID explicitly.
        label_id_str = str(top1["label"]).replace("LABEL_", "")
        try:
            label_id = int(label_id_str)
            label_str = id2label[label_id]
        except:
            # Fallback if config is different
            label_str = top1["label"]
            
        predicted_labels.append(label_str)
        confidences.append(top1["score"])
        margins.append(top1["score"] - top2_score)
        
    chunk_res = pd.DataFrame({
        "comment_id": ids,
        "predicted_label": predicted_labels,
        "confidence": confidences,
        "margin": margins
    })
    
    # Save checkpoint
    cp_path = os.path.join(CHECKPOINT_DIR, f"batch_{int(time.time())}.parquet")
    chunk_res.to_parquet(cp_path, index=False)
    print(f"Saved checkpoint: {cp_path} (Took {time.time() - start_time:.2f}s)")

print("\nInference loop complete.")


# SECTION 8: Combine Checkpoints

In [ ]:
all_cps = glob.glob(os.path.join(CHECKPOINT_DIR, "batch_*.parquet"))
df_preds = pd.concat([pd.read_parquet(cp) for cp in all_cps], ignore_index=True)

df_final = df_corpus.merge(df_preds, on="comment_id", how="inner")
print(f"Final merged dataset size: {len(df_final):,} / {len(df_corpus):,}")

assert len(df_final) == len(df_corpus), "Mismatch between output size and input corpus size!"

df_final.to_parquet(INFERENCE_OUTPUT, index=False)
print(f"Saved full corpus predictions to {INFERENCE_OUTPUT}")


# SECTION 9: Semantic & Statistical Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as plt_sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Label Distribution
dist = df_final["predicted_label"].value_counts()
print(dist)

fig, ax = plt.subplots(figsize=(10, 6))
dist.plot(kind="bar", color="#2b5c8f", ax=ax)
ax.set_title("Discourse Act Distribution in Full Corpus")
ax.set_ylabel("Count")
plt.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "01_label_distribution.png"), dpi=300)
plt.show()


In [ ]:
# Confidence vs Margin
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_final["confidence"].head(10000), df_final["margin"].head(10000), alpha=0.3, color="#7570b3")
ax.set_title("Confidence vs Margin (Sample 10k)")
ax.set_xlabel("Confidence (Top 1 Score)")
ax.set_ylabel("Margin (Top 1 - Top 2)")
plt.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "02_confidence_margin.png"), dpi=300)
plt.show()
